# 08 Split Into One Word Per Row

Reads `processed/07_stopwords.csv` and writes one row per person, cue and word, numbered in the order the person listed them ("bag doritos chips" gives three rows, positions 1, 2, 3).

- `position` counts the words that are left after the earlier steps ("has four legs" becomes `four` = 1, `leg` = 2).
- If a person writes the same word twice in one answer, only the first one is kept. `times_in_answer` says how many times it appeared.
- Answers with no words left are not in the file; they are listed in `processed/08_empty_answers.csv`.

Columns:

| Column | Meaning |
|---|---|
| `row_id` | row in `raw_data_all/all_data.csv` (links back to the original answer and source file) |
| `subject_id` | person (unique across all data) |
| `cue` | the cue word |
| `position` | order of the word in that person's answer for that cue |
| `word` | the word after all processing (lemma) |
| `original_word` | the word as the person typed it (after spell check and abbreviations) |
| `pos` | part of speech |
| `times_in_answer` | how many times the word appeared in the answer |
| `n_answered` | number of people who answered this cue (before any exclusions) |
| `n_final` | number of people who still have at least one word for this cue |

Output: `processed/08_words.csv`, and `processed/08_cue_counts.csv` (`n_answered`, `n_final` and `n_removed` for every cue).

In [1]:
import pandas as pd
from collections import Counter

In [2]:
data = pd.read_csv("processed/07_stopwords.csv", dtype=str, keep_default_na=False)
print(f"{len(data):,} answers")

185,494 answers


In [3]:
rows = []
empty = []

for row_id, subject_id, cue, tokens in zip(data["row_id"], data["subject_id"], data["cue"], data["tokens"]):
    items = [item.rsplit("|", 2) for item in tokens.split()]
    if not items:
        empty.append((row_id, subject_id, cue))
        continue
    times = Counter(lemma for _, lemma, _ in items)
    seen = set()
    position = 0
    for text, lemma, pos in items:
        if lemma in seen:
            continue
        seen.add(lemma)
        position += 1
        rows.append((row_id, subject_id, cue, position, lemma, text, pos, times[lemma]))

words = pd.DataFrame(rows, columns=["row_id", "subject_id", "cue", "position", "word",
                                    "original_word", "pos", "times_in_answer"])
print(f"{len(words):,} rows, {len(empty):,} answers with no words left")

952,792 rows, 1,082 answers with no words left


In [4]:
# how many people there were for each cue, before and after all the exclusions
all_data = pd.read_csv("raw_data_all/all_data.csv", usecols=["subject_id", "word", "wiki_rule"], dtype=str, keep_default_na=False)
all_data = all_data[~all_data["wiki_rule"].str.contains("stray_cue")]   # wrapped lines read as cues, not real cues
counts = pd.concat([
    all_data.groupby("word")["subject_id"].nunique().rename("n_answered"),
    words.groupby("cue")["subject_id"].nunique().rename("n_final"),
], axis=1).fillna(0).astype(int)
counts.index.name = "cue"
counts["n_removed"] = counts["n_answered"] - counts["n_final"]
counts.to_csv("processed/08_cue_counts.csv")
words = words.merge(counts[["n_answered", "n_final"]], left_on="cue", right_index=True, how="left")

pd.DataFrame(empty, columns=["row_id", "subject_id", "cue"]).to_csv("processed/08_empty_answers.csv", index=False)
words.to_csv("processed/08_words.csv", index=False)
print(f"Saved {len(words):,} rows to processed/08_words.csv")

Saved 952,792 rows to processed/08_words.csv


## Checks

In [5]:
print(f"{words['subject_id'].nunique():,} people, {words['cue'].nunique():,} cues, {words['word'].nunique():,} different words")
print(f"words per answer: mean {len(words) / words[['row_id']].nunique().iloc[0]:.1f}")
print(f"repeated words removed: {(words['times_in_answer'] - 1).sum():,}")
words.groupby("cue").size().describe()

13,629 people, 3,722 cues, 29,913 different words
words per answer: mean 5.2
repeated words removed: 33,491


count    3722.000000
mean      255.989253
std       181.719318
min        58.000000
25%       131.000000
50%       180.000000
75%       329.000000
max      2267.000000
dtype: float64

In [6]:
words[words["cue"] == "sidewalk"].head(12)

,row_id,subject_id,cue,position,word,original_word,pos,times_in_answer,n_answered,n_final
255650,57393,1072,sidewalk,1,road,road,PROPN,1,30,30
255651,57393,1072,sidewalk,2,pedestrian,pedestrian,NOUN,1,30,30
255652,57393,1072,sidewalk,3,cement,cement,NOUN,1,30,30
255674,57398,1123,sidewalk,1,walk,walking,VERB,1,30,30
255675,57398,1123,sidewalk,2,next,next,ADV,1,30,30
255676,57398,1123,sidewalk,3,road,roads,NOUN,1,30,30
255677,57398,1123,sidewalk,4,good,good,ADJ,1,30,30
255678,57398,1123,sidewalk,5,pedestrian,pedestrians,NOUN,1,30,30
255679,57398,1123,sidewalk,6,often,often,ADV,1,30,30
255680,57398,1123,sidewalk,7,concrete,concrete,NOUN,1,30,30


In [7]:
words["word"].value_counts().head(25)

word
people       6650
person       5109
one          4530
good         4469
food         4200
time         3943
place        3720
water        3599
small        3222
animal       3108
part         2971
money        2792
bad          2711
work         2650
hard         2573
color        2562
different    2472
body         2455
long         2437
many         2404
large        2293
take         2264
type         2175
big          2157
come         2129
Name: count, dtype: int64